## Feature processing for timeseries analysis

- This notebook is focused on analyzing the mean behavior of the library, over time

- It outputs a timeseries dataframe and a steady state dataframe

- Most steady state analysis is in another notebook, however

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import scipy.stats
from scipy.stats import ttest_ind

import sklearn as skl
import dask.dataframe as dd
import dask.array as da
import dask
import warnings
import copy
import random
from sklearn.metrics.pairwise import euclidean_distances,manhattan_distances,cosine_distances
from matplotlib.lines import Line2D

from sklearn.metrics import silhouette_score
import statsmodels.stats.multitest
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AffinityPropagation
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering
from dask.distributed import wait

from matplotlib import pyplot as plt
import ast


import pylab
import scipy.cluster.hierarchy as sch

import matplotlib.gridspec as gridspec
import matplotlib as mpl

import holoviews as hv
hv.extension('bokeh')
dask_random_seed = 42
random.seed(42)
np.random.seed(42)

warnings.filterwarnings(action='once',category=UserWarning)

In [ ]:
def get_sgrnadf_from_scoredf(scoredf, feature_labels, time_label="final cell timepoints list", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    scoredf_groupby = scoredf.groupby("sgRNA")
    sgrnadf = scoredf_groupby.apply(lambda x: x[trenchid_label].tolist()).to_frame().rename(columns={0:trenchid_label})
    
    for feature_label in feature_labels:    
        sgrnadf[feature_label + ": score"] = scoredf_groupby.apply(lambda x: np.array([val for item in x[feature_label + ": score"].tolist() for val in item]))
        
    sgrnadf[time_label] = scoredf_groupby.apply(lambda x: np.array([val for item in x[time_label].tolist() for val in item]))
    sgrnadf["Gene"] = scoredf_groupby.apply(lambda x: x["Gene"].iloc[0])
    sgrnadf["TargetID"] = scoredf_groupby.apply(lambda x: x["TargetID"].iloc[0])
    sgrnadf["N Mismatch"] = scoredf_groupby.apply(lambda x: x["N Mismatch"].iloc[0])
    sgrnadf["N Observations"] = scoredf_groupby.apply(lambda x: len(x[trenchid_label].tolist()))
    sgrnadf["Category"] = scoredf_groupby.apply(lambda x: x["Category"].iloc[0])
        
    return sgrnadf

def normalize_timeseries(feature_vector_series, lmbda=0.5):
    timeseries_arr = np.swapaxes(np.array(feature_vector_series.tolist()),1,2)
    sigma = np.std(timeseries_arr,axis=1)
    if lmbda>0.:
        sigma_prime = ((sigma + 1)**lmbda - 1) / lmbda ##yeo-johnson
    elif lmbda==0.:
        sigma_prime = np.log(sigma + 1)
    else:
        raise ValueError("lmbda cannot be negative")
    normalizer = sigma/sigma_prime
    normalized_timeseries = timeseries_arr/normalizer[:,np.newaxis,:]
    return normalized_timeseries

def get_timepoint_values(df, label, min_time, max_time, time_label="time (s)", trenchid_label="Multi-Experiment Phenotype Trenchid", flatten_vals=True):
    if flatten_vals:
        masked_label_series = df.apply(lambda x: np.array(x[label])[(np.array(x[time_label])>=min_time)*(np.array(x[time_label])<=max_time)], axis=1, meta="object")
        concat_series = masked_label_series.map_partitions(lambda x: np.concatenate(x.tolist()), meta=("Concat",object)).persist()
        delayed_series = concat_series.to_delayed()
        concat_da = [da.from_delayed(item, shape=(len(item.compute()),), dtype=float) for item in delayed_series]
        flattened_vals = da.concatenate(concat_da, axis=0).compute()        
        return flattened_vals
    else:
        masked_label_series = df.groupby(trenchid_label).apply(lambda x: np.array(x[label])[(np.array(x[time_label])>=min_time)*(np.array(x[time_label])<=max_time)], meta="object").persist()
        return masked_label_series

def get_feature_stats(df, feature_label, min_time, max_time,time_label="time (s)",trenchid_label="Multi-Experiment Phenotype Trenchid"):
    feature_vals = get_timepoint_values(df, feature_label, min_time, max_time,time_label=time_label,trenchid_label=trenchid_label)
    feature_median = np.median(feature_vals)
    feature_iqr = sp.stats.iqr(feature_vals)
    return feature_median,feature_iqr

def compute_score(df, feature_label, feature_median, feature_iqr):
    scores = 1.35*((df[feature_label] - feature_median)/feature_iqr)
    return scores

def get_feature_scores(df, feature_label, init_time_range=(0,28800), time_label="time (s)", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    feature_median,feature_iqr = get_feature_stats(df,feature_label,init_time_range[0],init_time_range[1],time_label=time_label,trenchid_label=trenchid_label)
    scores = compute_score(df, feature_label, feature_median, feature_iqr)
    return scores

def get_all_feature_scores(df, feature_labels, init_time_range=(0,28800), clipping_range=(-4,4), time_label="time (s)", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    
    for feature_label in feature_labels:
        print(feature_label)
        feature_scores = get_feature_scores(df, feature_label, init_time_range=init_time_range, time_label=time_label, trenchid_label=trenchid_label)
        if clipping_range != None:
            feature_scores = np.clip(feature_scores,clipping_range[0],clipping_range[1])
        df[feature_label + ": z score"] = feature_scores.persist()
    
    return df

def get_l_norm(x,right_tail_only=False):
    if right_tail_only:
        x_median = np.nanmedian(x)
        x_vals = x[x>x_median]
    else:
        x_vals = x
    l_norm = sp.stats.yeojohnson_normmax(x_vals)
    return l_norm

def apply_transform(final_output_df_filtered,yeo_subsample = 0.05,dask_random_seed=dask_random_seed,early_time_cutoff=28800,steady_state_time_cutoff=64800,clipping_range=(-4,4),\
                       params_to_transform = ['Division: major_axis_length'], transform_list=["YJ-Right"], time_label="time (s)",trenchid_label="Multi-Experiment Phenotype Trenchid"):
    
    subsample_df = final_output_df_filtered.sample(frac=yeo_subsample,random_state=dask_random_seed).persist()
    subsample_df = subsample_df[subsample_df[time_label]>=steady_state_time_cutoff]
    for i,param in enumerate(params_to_transform):
        print(param)
        transform = transform_list[i]
        all_param_values = subsample_df[param].astype(float).compute().tolist()
        all_param_values = np.array(all_param_values)
        all_param_values = all_param_values[~np.isnan(all_param_values)]
        
        if transform == "YJ" or transform == "YJ-Right":
            if transform == "YJ":
                l_norm = get_l_norm(all_param_values,right_tail_only=False)
            elif transform == "YJ-Right":
                l_norm = get_l_norm(all_param_values,right_tail_only=True)
            final_output_df_filtered[param + ": Transformed"] = final_output_df_filtered[param].apply(lambda x: sp.stats.yeojohnson(x,lmbda = l_norm), meta=(param + ": Transformed",float)).persist()
        elif transform == "Quant":
            quant_transformer = skl.preprocessing.QuantileTransformer(n_quantiles=1000,output_distribution='normal',subsample=len(all_param_values))
            quant_transformer = quant_transformer.fit(all_param_values[:,np.newaxis])
            final_output_df_filtered[param + ": Transformed"] = final_output_df_filtered[param].apply(lambda x: \
                        quant_transformer.transform(np.array(x).reshape(-1,1))[0,0], meta=(param + ": Transformed",float)).persist()
        elif transform == "None": # If I don't want to transform it
            final_output_df_filtered[param + ": Transformed"] = final_output_df_filtered[param].persist()

    final_output_df_filtered = get_all_feature_scores(final_output_df_filtered, [param + ": Transformed" for param in params_to_transform],\
                                                      init_time_range=(0, early_time_cutoff),clipping_range=clipping_range,\
                                                      time_label=time_label,trenchid_label=trenchid_label)
    final_output_df_filtered = final_output_df_filtered.reset_index().set_index(trenchid_label,drop=True)
    return final_output_df_filtered

def apply_transform_v2(final_output_df_filtered,yeo_subsample = 0.05,early_time_cutoff=28800,steady_state_time_cutoff=64800,clipping_range=(-4,4),\
                       params_to_transform = ['Division: major_axis_length'], transform_list=["YJ-Right"], time_label="time (s)",trenchid_label="Multi-Experiment Phenotype Trenchid"):
    
    params_to_transform_z_score = [param + ": z score" for param in params_to_transform]
    final_output_df_filtered = get_all_feature_scores(final_output_df_filtered, params_to_transform,init_time_range=(0, early_time_cutoff),\
                                                      clipping_range=None,time_label=time_label,trenchid_label=trenchid_label)
    
    subsample_df = final_output_df_filtered.sample(frac=yeo_subsample,random_state=dask_random_seed).persist()
    subsample_df = subsample_df[subsample_df[time_label]>=steady_state_time_cutoff]
    for i,param in enumerate(params_to_transform_z_score):
        print(param)
        transform = transform_list[i]
        all_param_values = subsample_df[param].astype(float).compute().tolist()
        all_param_values = np.array(all_param_values)
        all_param_values = all_param_values[~np.isnan(all_param_values)]
        
        if transform == "YJ" or transform == "YJ-Right":
            if transform == "YJ":
                l_norm = get_l_norm(all_param_values,right_tail_only=False)
            elif transform == "YJ-Right":
                l_norm = get_l_norm(all_param_values,right_tail_only=True)
            final_output_df_filtered[param + ": Transformed"] = final_output_df_filtered[param].apply(lambda x: sp.stats.yeojohnson(x,lmbda = l_norm), meta=(param + ": Transformed",float))
            final_output_df_filtered[param + ": Transformed"] = np.clip(final_output_df_filtered[param + ": Transformed"],clipping_range[0],clipping_range[1]).persist()
        elif transform == "Quant":
            quant_transformer = skl.preprocessing.QuantileTransformer(n_quantiles=1000,output_distribution='normal',subsample=len(all_param_values))
            quant_transformer = quant_transformer.fit(all_param_values[:,np.newaxis])
            final_output_df_filtered[param + ": Transformed"] = final_output_df_filtered[param].apply(lambda x: \
                        quant_transformer.transform(np.array(x).reshape(-1,1))[0,0], meta=(param + ": Transformed",float))
            final_output_df_filtered[param + ": Transformed"] = np.clip(final_output_df_filtered[param + ": Transformed"],clipping_range[0],clipping_range[1]).persist()
        elif transform == "None":
            final_output_df_filtered[param + ": Transformed"] = np.clip(final_output_df_filtered[param],clipping_range[0],clipping_range[1]).persist()
    final_output_df_filtered = final_output_df_filtered.reset_index().set_index(trenchid_label,drop=True)
    return final_output_df_filtered

### Initial Data Processing (Mean Timeseries)

- Here I use the following method to standardize data
    1) Raw data from the steady state period of the experiment are taken and Yeo-Johnson Transformed
    2) Then, values are standardized by the uninduced distribution
- Note for the Y-J transform, I only use the right skew of the distribution, to force attention to the right tail of the distribution
- Note that, for the final version of this analysis, I omit quantile transforms since they are difficult to explain and do not add much

### Thoughts

1) I don't like that the Y-J transform is acting on random raw values, this distribution is very noisey and does not represent the variability that will need to be standardized.
2) However, applying standardization to the aggregation across trenches also seems wrong, since outliers will be further upweighted.
3) A resonable middle ground appears (to me):
    - Produce a smoothed trace per trench (without any transformation/standardization)
    - Transform and standardize using the distributions of these trench-aggregated values
    - Then, aggregate these z-scores by variant
    - If this also gives bad results, I can try further aggregation (keep refering to the steady state ipynb where this worked well)


#### Start Dask

In [ ]:
# dask_controller = tr.trcluster.dask_controller(
#     walltime="04:00:00",
#     local=False,
#     n_workers=200,
#     n_workers_min=200,
#     memory="16GB",
#     working_directory="/home/de64/scratch/de64/dask",
# )
# dask_controller.startdask()

dask_controller = tr.trcluster.dask_controller(
    walltime="04:00:00",
    local=False,
    n_workers=100,
    n_workers_min=30,
    memory="16GB",
    working_directory="/home/lag36/scratch/lag36/dask",
)
dask_controller.startdask()

In [ ]:
dask_controller.daskclient

### Kernel regression on each trench

In [ ]:
from statsmodels.nonparametric import kernel_regression
from scipy.stats import iqr
from statsmodels.nonparametric.smoothers_lowess import lowess
import sklearn as skl
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance


def timeseries_kernel_reg(df, y_label, min_tpt, max_tpt, kernel_bins, kernel_bandwidth, time_label="Final time (s)", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    def kernel_reg(x_arr,y_arr,start=min_tpt,end=max_tpt,kernel_bins=kernel_bins,kernel_bandwidth=kernel_bandwidth):
        intervals = np.linspace(start, end, num=kernel_bins, dtype=float)
        w = kernel_regression.KernelReg(y_arr,x_arr,"c",reg_type="lc",bw=np.array([kernel_bandwidth]),ckertype="gaussian").fit(intervals)[0]
        reg_x, reg_y = (intervals, w)
        return reg_x, reg_y
    
    # if time_label == "Cell Cycle":
    #     # if this is the cell cycle df, do the following
    #     kernel_result = df.groupby(trenchid_label).apply(lambda x: kernel_reg((x["Final time (s)"].values - (x["Delta time (s)"].values/2)),x[y_label].values,)[1],meta=(y_label, object),)
    # else:
    kernel_result = df.groupby(trenchid_label).apply(lambda x: kernel_reg(x[time_label].values,x[y_label].values,)[1],meta=(y_label, object),)
    return kernel_result


def get_all_kernel_regs(df, y_label_list, min_tpt = 0, max_tpt = 79200, kernel_bins=44, kernel_bandwidth = 2700, time_label="Cell Cycle", trenchid_label="Multi-Experiment Phenotype Trenchid"):
    out_df = copy.copy(df)

    for y_label in y_label_list:
        kernel_result = timeseries_kernel_reg(out_df, y_label, min_tpt, max_tpt, kernel_bins, kernel_bandwidth, time_label=time_label, trenchid_label=trenchid_label)
        out_df["Kernel Trace: " + y_label] = kernel_result.persist()
        
    return out_df

In [ ]:
cell_cycle_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Cell_Cycle_Merged/", engine="pyarrow",calculate_divisions=True)
timepoints_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Observations_Merged/", engine="pyarrow",calculate_divisions=True)
growth_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Growth_Observations_Merged/", engine="pyarrow",calculate_divisions=True)

In [ ]:
# Parameters I want traces for. Not necesarily to do dimensionality reduction. Keep these and add my own
cell_cycle_params = ['Birth: major_axis_length', 'Division: major_axis_length',
   'Delta: major_axis_length', 'Birth: minor_axis_length',
   'Division: minor_axis_length', 'Delta: minor_axis_length',
   'Birth: Volume', 'Division: Volume', 'Delta: Volume',
   'Final timepoints', 'Delta Timepoints', 'Final time (s)',
   'Delta time (s)','Septum Displacement','Septum Displacement Length Normalized',
                    'Birth: Number of Nucleoids',
'Division: Number of Nucleoids',
'Birth: Total Nucleoid Area',
'Division: Total Nucleoid Area',
'Birth: Nucleoid Area Mean',
'Division: Nucleoid Area Mean',
'Birth: Nucleoid Major Axis Length Mean',
'Division: Nucleoid Major Axis Length Mean',
'Birth: Nucleoid Minor Axis Length Mean',
'Division: Nucleoid Minor Axis Length Mean']

timepoints_params = ['area','major_axis_length', 'minor_axis_length', 'Volume','mCherry mean_intensity',
                     'YFP mean_intensity','Number of Nucleoids', 'Total Nucleoid Area',
'Nucleoid Area Mean',
'Nucleoid Mean Intensity Mean',
'Nucleoid Major Axis Length Mean',
'Nucleoid Minor Axis Length Mean',
'Nucleoid Separation Mean']

growth_params = ['Instantaneous Growth Rate: Volume', 'Instantaneous Growth Rate: major_axis_length']
all_traced_params = cell_cycle_params + timepoints_params + growth_params
all_traced_params = ["Kernel Trace: " + item for item in all_traced_params]
bandwidth_hours = 2 # Keep the same (bandwidth of Gaussian for 

cell_cycle_df = get_all_kernel_regs(
    cell_cycle_df,
    cell_cycle_params,
    min_tpt = 0, # Seconds
    max_tpt = 36000, # Seconds 36000 for 10 hours
    kernel_bins = 20, # Number of points in my time series (Set to 20 i.e. 2 per hour)
    kernel_bandwidth = bandwidth_hours*3600,
    time_label="Mid-Cycle time (s)",
    trenchid_label="Multi-Experiment Phenotype Trenchid")

timepoints_df = get_all_kernel_regs(
    timepoints_df,
    timepoints_params,
    min_tpt = 0,
    max_tpt = 36000,
    kernel_bins = 20,
    kernel_bandwidth = bandwidth_hours*3600,
    time_label="Observation time (s)",
    trenchid_label="Multi-Experiment Phenotype Trenchid")

growth_df = get_all_kernel_regs(
    growth_df,
    growth_params,
    min_tpt = 0,
    max_tpt = 36000,
    kernel_bins = 20,
    kernel_bandwidth = bandwidth_hours*3600,
    time_label="Measurement time (s)",
    trenchid_label="Multi-Experiment Phenotype Trenchid")

#### Merge Timeseries dfs

In [ ]:
wait(cell_cycle_df);
wait(timepoints_df);
wait(growth_df);

In [ ]:
trenchid_label="Multi-Experiment Phenotype Trenchid"

cell_cycle_out = cell_cycle_df.groupby(trenchid_label).apply(lambda x: x.iloc[0])
timepoints_out = timepoints_df.groupby(trenchid_label).apply(lambda x: x.iloc[0])
growth_out = growth_df.groupby(trenchid_label).apply(lambda x: x.iloc[0])

traced_col_names = [col_name for col_name in cell_cycle_out.columns.tolist() if col_name in all_traced_params]
cell_cycle_out = cell_cycle_out[['Global CellID', 'File Parquet Index', 'fov', 'row', 'trench',
       'initial timepoints', 'File Index', 'File Trench Index', 'CellID',
       'Trench Score', 'Mother CellID', 'Daughter CellID 1',
       'Daughter CellID 2', 'Sister CellID', 'Centroid X', 'Centroid Y',
       'Kymograph File Parquet Index', 'Kymograph FOV Parquet Index',
       'FOV Parquet Index', 'Experiment #', 'phenotype trenchid'] + traced_col_names]

traced_col_names = [col_name for col_name in timepoints_out.columns.tolist() if col_name in all_traced_params]
timepoints_out = timepoints_out[traced_col_names]

traced_col_names = [col_name for col_name in growth_out.columns.tolist() if col_name in all_traced_params]
growth_out = growth_out[traced_col_names]

timeseries_merged_dd = dd.concat([cell_cycle_out,timepoints_out,growth_out], axis=1)
timeseries_merged_dd = timeseries_merged_dd.persist()
wait(timeseries_merged_dd);

#### Export Kernel Regression Dataframe

In [ ]:
kernel_trace_columns = [column_name for column_name in timeseries_merged_dd.columns if "Kernel Trace" in column_name]
explode_series_list = []
for column_name in kernel_trace_columns:
    working_exploded_column = timeseries_merged_dd[column_name].explode().astype(float).persist()
    explode_series_list.append(working_exploded_column)
kernel_output_df = dd.concat(explode_series_list, axis=1).persist()
wait(kernel_output_df);

n_indices = len(timeseries_merged_dd)
first_kernel_trace_column = [column_name for column_name in timeseries_merged_dd.columns if "Kernel Trace" in column_name][0]
timeseries_len = len(timeseries_merged_dd[first_kernel_trace_column].get_partition(0).compute().iloc[0])
timepoint_index = np.tile(range(timeseries_len),n_indices)
kernel_output_df = tr.add_list_to_column(kernel_output_df,list(timepoint_index),"Kernel Timepoints").persist()
wait(kernel_output_df);

kernel_output_df = kernel_output_df.reset_index()
kernel_output_df["Multi-Experiment Phenotype Trenchid-Kernel Timepoint Index"] =  kernel_output_df.apply(lambda x: int(f'{x["Multi-Experiment Phenotype Trenchid"].astype(int):010n}{x["Kernel Timepoints"]:03n}'), axis=1, meta=(None,'int64')).persist()
kernel_output_df = kernel_output_df.set_index("Multi-Experiment Phenotype Trenchid-Kernel Timepoint Index",sorted=True)
wait(kernel_output_df);

kernel_output_df.to_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-06_lDE25_Kernel_Regression_df",\
                            engine="pyarrow",schema="infer",overwrite=True)
dask_controller.daskclient.cancel(kernel_output_df)

### Compute Yeo-Johnson Transform, Standardize Data and Save Checkpoint

In [ ]:
# These are the ones in which I will do dim reduction on. 
cell_cycle_params_to_transform=['Delta time (s)','Septum Displacement Length Normalized']
timepoints_params_to_transform=['major_axis_length','minor_axis_length','mCherry mean_intensity',
                                'Number of Nucleoids',
'YFP mean_intensity',
'Total Nucleoid Area',
'Nucleoid Area Mean',
'Nucleoid Mean Intensity Mean',
'Nucleoid Major Axis Length Mean',
'Nucleoid Minor Axis Length Mean']#,
# 'Nucleoid Separation Mean']

growth_params_to_transform=['Instantaneous Growth Rate: Volume']
params_to_transform_kernel_reg = ["Kernel Trace: " + item for item in (cell_cycle_params_to_transform+timepoints_params_to_transform+growth_params_to_transform)]

# YJ by default. Consider YJ-Right if heavy positive tail
# Quant: Forces Gaussian regardless of shape
transform_list = ["YJ-Right", #Delta time (s)
                  "YJ", # Septum displacement length
                  "Quant", # Major
                  "Quant", # Minor
                  "YJ-Right", # mCherry mean_int
                  "Quant", #N Nucs
                  "YJ-Right", # YFP mean int
                  "Quant", # Total nuc Area
                  "Quant", # Nuc Area Mean
                  "YJ-Right", #Nuc Mean Int Mean
                  "Quant",#Nuc Major Axis Mean
                  "Quant",# Nuc Minor Axis Mean
                  # "None",# Nuc Sep Mean
                  "Quant"] # Growth Rate:Volume

In [ ]:

test = akernel_output_df.get_partition(0).compute()

In [ ]:
timepoint_step

In [ ]:
early_timepoint_cutoff

61 frames
10 minutes/frame

10*60 = 600 minutes total
10 hours total

max_tpts = 36,000 sec


In [ ]:
steady_state_timepoint_cutoff

In [ ]:
8*3600

In [ ]:
timepoint_step

In [ ]:
steady_state_time

In [ ]:
# Same as above
kernel_bins = 20
min_tpt = 0
max_tpt = 36000
induction_time = 2*3600 #2*3600
steady_state_time = 8*3600 #2 hours before the end 
timepoint_step = (max_tpt-min_tpt)/kernel_bins
early_timepoint_cutoff = int(induction_time//timepoint_step)
steady_state_timepoint_cutoff = int(steady_state_time//timepoint_step)

kernel_output_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-06_lDE25_Kernel_Regression_df",engine="pyarrow",calculate_divisions=True)
kernel_output_transform_df = apply_transform(kernel_output_df,yeo_subsample = 0.01,early_time_cutoff=early_timepoint_cutoff,steady_state_time_cutoff=steady_state_timepoint_cutoff,\
                    clipping_range=(-4,4),params_to_transform = params_to_transform_kernel_reg, transform_list = transform_list,\
                                         time_label="Kernel Timepoints",trenchid_label="Multi-Experiment Phenotype Trenchid")
kernel_output_transform_df = kernel_output_transform_df.persist()
wait(kernel_output_transform_df);
kernel_output_transform_df.to_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-06_lDE25_Kernel_Regression_Transform_df", engine="pyarrow", overwrite=True)
dask_controller.daskclient.cancel(kernel_output_df)
dask_controller.daskclient.cancel(kernel_output_transform_df)
dask_controller.reset_worker_memory()

### Plot Transformed Features

In [ ]:
# These are the ones in which I will do dim reduction on. 
cell_cycle_params_to_transform=['Delta time (s)','Septum Displacement Length Normalized']
timepoints_params_to_transform=['major_axis_length','minor_axis_length','mCherry mean_intensity',
                                'Number of Nucleoids',
'YFP mean_intensity',
'Total Nucleoid Area',
'Nucleoid Area Mean',
'Nucleoid Mean Intensity Mean',
'Nucleoid Major Axis Length Mean',
'Nucleoid Minor Axis Length Mean']#,
# 'Nucleoid Separation Mean']

growth_params_to_transform=['Instantaneous Growth Rate: Volume']
params_to_transform_kernel_reg = ["Kernel Trace: " + item for item in (cell_cycle_params_to_transform+timepoints_params_to_transform+growth_params_to_transform)]

cell_cycle_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Cell_Cycle_Merged/", engine="pyarrow",calculate_divisions=True)
kernel_output_yj_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-06_lDE25_Kernel_Regression_Transform_df", engine="pyarrow",calculate_divisions=True)

In [ ]:
sample_frac = 0.01
kernel_output_yj_df_sampled = kernel_output_yj_df.sample(frac=sample_frac,random_state=dask_random_seed).compute()

In [ ]:
ncols=3
nrows=10

# Change names here
param_names = ["Interdivision Time (hr)",
               "Septum Displacement",
               r"Length ($ \mu m $)",
               r"Width ($ \mu m $)",
               "mCherry Intensity (AU)",
               "Number of Nucleoids",
               "YFP Intensity (AU)",
               "Total Nucleoid Area",
               "Mean Nucleoid Area",
               "Nucleoid Mean Intensity",
               "Nucleoid Major Axis Length",
               "Nucleoid Minor Axis Length",
               "Growth Rate (1/hr)"]
param_ranges = [(0,5000),
                (0,0.05),
                (0.,10.),
                (1.1,1.7),
                (0,10000), # mCherry Int
                (0, 5), # Number of nucs
                (0, 13000), # YFP Int
                (0, 8), # Total Area
                (0,8), # Mean Nuc Area
                (0, 13000), # NUc Mean Int
                (0,10), # Nuc Major
                (0.6, 1.5), # Nuc Minor
                (0,2)] # Growth

# Change the grid
fig = plt.figure(figsize=(8,16))
for i,param in enumerate(params_to_transform_kernel_reg):
    working_row = i//ncols
    param_name = param_names[i]
    param_range = param_ranges[i]
    param_transformed = param + ": Transformed: z score"
    vals = kernel_output_yj_df_sampled[param].values
    transformed_vals = kernel_output_yj_df_sampled[param_transformed].values
    
    plt.subplot(nrows,ncols,i+1)
    if i == 1:
        plt.title("Raw Values",fontsize=16)
    plt.hist(vals,bins=50,range=param_range)
    plt.xlabel(param_name)
    plt.subplot(nrows,ncols,i+1+(5*ncols))
    if i == 1:
        plt.title("Z-Scores",fontsize=16)
    plt.hist(transformed_vals,bins=50,range=(-5,5))
    plt.xlabel(param_name)
plt.tight_layout()
plt.savefig("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-06_Timeseries_Zscore_Transformation.png",dpi=300)

#### Merge Timeseries dfs

In [ ]:
trenchid_label="Multi-Experiment Phenotype Trenchid"
kernel_score_params = [param + ": Transformed: z score" for param in params_to_transform_kernel_reg]
all_kernel_reg_columns = all_traced_params + kernel_score_params

kernel_output_yj_df_unexploded = kernel_output_yj_df.groupby(trenchid_label).apply(lambda x: pd.DataFrame({column: [np.array(x[column].tolist())] for column in x.columns}))
kernel_output_yj_df_unexploded = kernel_output_yj_df_unexploded[all_kernel_reg_columns]
kernel_output_yj_df_unexploded = kernel_output_yj_df_unexploded.persist()
wait(kernel_output_yj_df_unexploded);

cell_cycle_out = cell_cycle_df.groupby(trenchid_label).apply(lambda x: x.iloc[0])
cell_cycle_out = cell_cycle_out[['Global CellID', 'File Parquet Index', 'fov', 'row', 'trench',
       'initial timepoints', 'File Index', 'File Trench Index', 'CellID',
       'Trench Score', 'Mother CellID', 'Daughter CellID 1',
       'Daughter CellID 2', 'Sister CellID', 'Centroid X', 'Centroid Y',
       'Kymograph File Parquet Index', 'Kymograph FOV Parquet Index',
       'FOV Parquet Index', 'Experiment #', 'phenotype trenchid']]
cell_cycle_out = cell_cycle_out.persist()
wait(cell_cycle_out);

kernel_output_yj_df_unexploded = kernel_output_yj_df_unexploded.reset_index().set_index("Multi-Experiment Phenotype Trenchid", sorted=True)
kernel_output_yj_df_unexploded = kernel_output_yj_df_unexploded.drop("level_1",axis=1).persist()
wait(kernel_output_yj_df_unexploded);

timeseries_merged_dd = cell_cycle_out.join(kernel_output_yj_df_unexploded,on="Multi-Experiment Phenotype Trenchid")

#### Average over trenches for sgRNA-wise Timeseries

In [ ]:
final_output_df_barcodes = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Final_Barcodes_df_Merged",engine="pyarrow",calculate_divisions=True)
final_output_df_barcodes = final_output_df_barcodes.groupby(trenchid_label,sort=False).apply(lambda x: x.iloc[0]).persist()
wait(final_output_df_barcodes);

final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index("oDEPool7_id",drop=True)
final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index(trenchid_label).persist()
wait(final_output_df_barcodes);
timeseries_merged_dd = timeseries_merged_dd.merge(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner',left_index=True,right_index=True).persist()

n_obs = timeseries_merged_dd.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs = pd.DataFrame(n_obs).rename({0:"N Observations"}, axis=1).sort_index()
timeseries_merged_dd = timeseries_merged_dd.merge(n_obs,on="oDEPool7_id",how='inner',right_index=True)

feature_vector_series = timeseries_merged_dd.apply(lambda x: np.array(x[kernel_score_params].tolist()), axis=1)
timeseries_merged_dd["Feature Vector"] = feature_vector_series

timeseries_merged_dd_nan_filtered = timeseries_merged_dd[~timeseries_merged_dd["Feature Vector"].apply(lambda x: np.any(np.isnan(x)),meta=("No NaN",bool))]

#### Aggregate over timeseries

In [ ]:
## params
vectors_to_aggregate = ['Feature Vector'] + all_kernel_reg_columns

timeseries_merged_dd_nan_filtered_sgRNA_sorted = timeseries_merged_dd_nan_filtered.set_index("oDEPool7_id").persist()
timeseries_merged_dd_nan_filtered_sgRNA_sorted_groupby = timeseries_merged_dd_nan_filtered_sgRNA_sorted.groupby(["oDEPool7_id"])
    
sgRNA_df_merged = timeseries_merged_dd_nan_filtered_sgRNA_sorted_groupby.apply(lambda x: x.iloc[0]).persist()
sgRNA_df_merged = sgRNA_df_merged.compute()

for vector in vectors_to_aggregate:
    print(vector)
    mean_vectors = timeseries_merged_dd_nan_filtered_sgRNA_sorted_groupby.apply(lambda x: np.mean(x[vector], axis=0), meta=(None,object)).compute()
    sgRNA_df_merged[vector] = mean_vectors
    sem_vectors = timeseries_merged_dd_nan_filtered_sgRNA_sorted_groupby.apply(lambda x: sp.stats.sem(np.array(list(x[vector])), axis=0, nan_policy='propagate'), meta=(None,object)).compute()
    sgRNA_df_merged["SEM: " + vector] = sem_vectors
sgRNA_df_merged.to_pickle("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-06_sgRNA_Timeseries_df.pkl")

In [ ]:
dask_controller.daskclient.cancel(cell_cycle_df)
dask_controller.daskclient.cancel(kernel_output_yj_df)
dask_controller.daskclient.cancel(timeseries_merged_dd)
dask_controller.daskclient.cancel(final_output_df_barcodes)
dask_controller.reset_worker_memory()

In [ ]:
dask_controller.shutdown()

#### Get Steady State Measurements

In [ ]:
cell_cycle_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Cell_Cycle_Merged/", engine="pyarrow",calculate_divisions=True)
timepoints_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Observations_Merged/", engine="pyarrow",calculate_divisions=True)
growth_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Growth_Observations_Merged/", engine="pyarrow",calculate_divisions=True)

cell_cycle_params = ['Birth: major_axis_length', 'Division: major_axis_length',
   'Delta: major_axis_length', 'Birth: minor_axis_length',
   'Division: minor_axis_length', 'Delta: minor_axis_length',
   'Birth: Volume', 'Division: Volume', 'Delta: Volume',
   'Final timepoints', 'Delta Timepoints', 'Final time (s)',
   'Delta time (s)','Septum Displacement','Septum Displacement Length Normalized',
                    'Birth: Number of Nucleoids',
'Division: Number of Nucleoids',
'Birth: Total Nucleoid Area',
'Division: Total Nucleoid Area',
'Birth: Nucleoid Area Mean',
'Division: Nucleoid Area Mean',
'Birth: Nucleoid Major Axis Length Mean',
'Division: Nucleoid Major Axis Length Mean',
'Birth: Nucleoid Minor Axis Length Mean',
'Division: Nucleoid Minor Axis Length Mean']

timepoints_params = ['area','major_axis_length', 'minor_axis_length', 'Volume','mCherry mean_intensity',
                     'YFP mean_intensity','Number of Nucleoids', 'Total Nucleoid Area',
'Nucleoid Area Mean',
'Nucleoid Mean Intensity Mean',
'Nucleoid Major Axis Length Mean',
'Nucleoid Minor Axis Length Mean',
'Nucleoid Separation Mean']

growth_params = ['Instantaneous Growth Rate: Volume', 'Instantaneous Growth Rate: major_axis_length']
all_traced_params = cell_cycle_params + timepoints_params + growth_params

#### Use Steady State Time to Filter

In [ ]:
steady_state_time = 8*3600 #18 hours
induction_time = 2*3600 #8 hours

trenchid_label="Multi-Experiment Phenotype Trenchid"

cell_cycle_df_ss = cell_cycle_df[cell_cycle_df["Initial time (s)"]>=steady_state_time]
cell_cycle_df_pre = cell_cycle_df[cell_cycle_df["Final time (s)"]<=induction_time]

timepoints_df_ss = timepoints_df[timepoints_df["Observation time (s)"]>=steady_state_time]
timepoints_df_pre = timepoints_df[timepoints_df["Observation time (s)"]<=induction_time]

growth_df_ss = growth_df[growth_df["Measurement time (s)"]>=steady_state_time]
growth_df_pre = growth_df[growth_df["Measurement time (s)"]<=induction_time]

#### Add sgRNA info for real data and save as checkpoint

In [ ]:
# cell_cycle_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Cell_Cycle_Merged/", engine="pyarrow",calculate_divisions=True)
# timepoints_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Observations_Merged/", engine="pyarrow",calculate_divisions=True)
# growth_df = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Lineage_Growth_Observations_Merged/", engine="pyarrow",calculate_divisions=True)

In [ ]:
final_output_df_barcodes = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-01-24_lDE25_Final_Barcodes_df_Merged",engine="pyarrow",calculate_divisions=True)
final_output_df_barcodes = final_output_df_barcodes.groupby(trenchid_label,sort=False).apply(lambda x: x.iloc[0]).persist()
wait(final_output_df_barcodes);

final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index("oDEPool7_id",drop=True)
final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index(trenchid_label).persist()
wait(final_output_df_barcodes);

cell_cycle_df_ss = cell_cycle_df_ss.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
cell_cycle_df_pre = cell_cycle_df_pre.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
n_obs = cell_cycle_df_ss.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs = pd.DataFrame(n_obs).rename({0:"N Observations"}, axis=1).sort_index()
cell_cycle_df_ss = cell_cycle_df_ss.merge(n_obs,on="oDEPool7_id",how='inner')
cell_cycle_df_pre = cell_cycle_df_pre.merge(n_obs,on="oDEPool7_id",how='inner')

In [ ]:
cell_cycle_df_ss.to_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Steady_State/", engine="pyarrow", overwrite=True)
cell_cycle_df_pre.to_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Preinduction/", engine="pyarrow", overwrite=True)
print("Cell Cycle Done.")

timepoints_df_ss = timepoints_df_ss.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
timepoints_df_pre = timepoints_df_pre.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
n_obs = timepoints_df_ss.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs = pd.DataFrame(n_obs).rename({0:"N Observations"}, axis=1).sort_index()
timepoints_df_ss = timepoints_df_ss.merge(n_obs,on="oDEPool7_id",how='inner')
timepoints_df_pre = timepoints_df_pre.merge(n_obs,on="oDEPool7_id",how='inner')
timepoints_df_ss.to_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Timepoints_Timeseries_Steady_State/", engine="pyarrow", overwrite=True)
timepoints_df_pre.to_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Timepoints_Timeseries_Preinduction/", engine="pyarrow", overwrite=True)
print("Timepoints Done.")

growth_df_ss = growth_df_ss.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
growth_df_pre = growth_df_pre.join(final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']],how='inner')
n_obs = growth_df_ss.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs = pd.DataFrame(n_obs).rename({0:"N Observations"}, axis=1).sort_index()
growth_df_ss = growth_df_ss.merge(n_obs,on="oDEPool7_id",how='inner')
growth_df_pre = growth_df_pre.merge(n_obs,on="oDEPool7_id",how='inner')
growth_df_ss.to_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Growth_Timeseries_Steady_State/", engine="pyarrow", overwrite=True)
growth_df_pre.to_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Growth_Timeseries_Preinduction/", engine="pyarrow", overwrite=True)
print("Growth Done.")
dask_controller.reset_worker_memory()

### Calculate Means, Variances and CVs of Measured Quantities

In [ ]:
def get_mean_var_and_cv(df,final_columns,columns_to_apply_aggregation,sgrna_key='oDEPool7_id'):
    sgrna_sorted = df.reset_index(drop=True).set_index(sgrna_key).persist()
    sgrna_groupby = sgrna_sorted.groupby(sgrna_key)
    mean_df = sgrna_groupby[columns_to_apply_aggregation].aggregate(np.mean).persist()
    std_df = sgrna_groupby[columns_to_apply_aggregation].aggregate(np.std).persist()
    cv_df = std_df/mean_df
    var_df = sgrna_groupby[columns_to_apply_aggregation].aggregate(np.var).persist()
    
    mean_df = mean_df.rename(columns={key: "Mean: " + key for key in mean_df.columns.tolist()})
    var_df = var_df.rename(columns={key: "Variance: " + key for key in var_df.columns.tolist()})
    cv_df = cv_df.rename(columns={key: "CV: " + key for key in cv_df.columns.tolist()})
    
    output_df = sgrna_groupby.first()
    output_df = output_df[final_columns]
    output_df = output_df.join(mean_df)
    output_df = output_df.join(var_df)
    output_df = output_df.join(cv_df)
    
    return output_df

In [ ]:
cell_cycle_df_ss = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Steady_State/",engine="pyarrow",calculate_divisions=True)
cell_cycle_df_pre = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Preinduction/",engine="pyarrow",calculate_divisions=True)

timepoints_df_ss = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Timepoints_Timeseries_Steady_State/",engine="pyarrow",calculate_divisions=True)
timepoints_df_pre = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Timepoints_Timeseries_Preinduction/",engine="pyarrow",calculate_divisions=True)

growth_df_ss = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Growth_Timeseries_Steady_State/",engine="pyarrow",calculate_divisions=True)
growth_df_pre = dd.read_parquet("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Growth_Timeseries_Preinduction/",engine="pyarrow",calculate_divisions=True)

final_columns = ['sgRNA','EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'N Observations']

cell_cycle_df_ss_mean_cv = get_mean_var_and_cv(cell_cycle_df_ss,final_columns,cell_cycle_params,sgrna_key='oDEPool7_id').compute()
timepoints_df_ss_mean_cv = get_mean_var_and_cv(timepoints_df_ss,final_columns,timepoints_params,sgrna_key='oDEPool7_id').compute()
growth_df_ss_mean_cv = get_mean_var_and_cv(growth_df_ss,final_columns,growth_params,sgrna_key='oDEPool7_id').compute()

Note by Luis: The cell below was modified to include the indices of the timepoints df instead of the growth df

In [ ]:
# timepoints_df_ss_mean_cv = timepoints_df_ss_mean_cv[["Mean: " + item for item in timepoints_params] + ["Variance: " + item for item in timepoints_params] + ["CV: " + item for item in timepoints_params]] # Daniel's Version
cell_cycle_df_ss_mean_cv = cell_cycle_df_ss_mean_cv[["Mean: " + item for item in cell_cycle_params] + ["Variance: " + item for item in cell_cycle_params] + ["CV: " + item for item in cell_cycle_params]] # Luis' Version

growth_df_ss_mean_cv = growth_df_ss_mean_cv[["Mean: " + item for item in growth_params] + ["Variance: " + item for item in growth_params] + ["CV: " + item for item in growth_params]]

# output_ss_meancv = pd.concat([cell_cycle_df_ss_mean_cv,timepoints_df_ss_mean_cv,growth_df_ss_mean_cv],axis=1).sort_index() # Daniel's Version
output_ss_meancv = pd.concat([timepoints_df_ss_mean_cv,growth_df_ss_mean_cv,cell_cycle_df_ss_mean_cv],axis=1).sort_index() # Luis's Version

In [ ]:
output_ss_meancv.to_pickle("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Steady_State_MeanCV.pkl")

del cell_cycle_df_ss_mean_cv
del timepoints_df_ss_mean_cv
del growth_df_ss_mean_cv
del output_ss_meancv

cell_cycle_df_pre_mean_cv = get_mean_var_and_cv(cell_cycle_df_pre,final_columns,cell_cycle_params,sgrna_key='oDEPool7_id').compute()
timepoints_df_pre_mean_cv = get_mean_var_and_cv(timepoints_df_pre,final_columns,timepoints_params,sgrna_key='oDEPool7_id').compute()
growth_df_pre_mean_cv = get_mean_var_and_cv(growth_df_pre,final_columns,growth_params,sgrna_key='oDEPool7_id').compute()

timepoints_df_pre_mean_cv = timepoints_df_pre_mean_cv[["Mean: " + item for item in timepoints_params] + ["Variance: " + item for item in timepoints_params] + ["CV: " + item for item in timepoints_params]]
growth_df_pre_mean_cv = growth_df_pre_mean_cv[["Mean: " + item for item in growth_params] + ["Variance: " + item for item in growth_params] + ["CV: " + item for item in growth_params]]

output_pre_meancv = pd.concat([cell_cycle_df_pre_mean_cv,timepoints_df_pre_mean_cv,growth_df_pre_mean_cv],axis=1).sort_index()
output_pre_meancv.to_pickle("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Preinduction_MeanCV.pkl")

In [ ]:
dask_controller.shutdown()

In [ ]:
df_pre = pd.read_pickle("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Preinduction_MeanCV.pkl")

In [ ]:
df_post = pd.read_pickle("/home/lag36/scratch/lag36/2023-01-08_lDE25_Run_2/2023-02-07_lDE25_Lineage_Cell_Cycle_Timeseries_Steady_State_MeanCV.pkl")

In [ ]:
ax = df_post.plot(x='Mean: major_axis_length', y='Mean: Total Nucleoid Area', marker = '.', linestyle = 'None')
df_pre.plot(x='Mean: major_axis_length', y='Mean: Total Nucleoid Area', marker = '.', linestyle = 'None', ax=ax)